In [3]:
# 필요한 라이브러리 설치
!pip install torch torchvision torchaudio
!pip install transformers
!pip install sentence-transformers
!pip install qdrant-client
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install openai
!pip install scikit-learn
!pip install rank_bm25
!pip install tiktoken
!pip install numpy
!pip install tqdm
!pip install langchain-qdrant
!pip install langchain-huggingface

In [4]:
import os
from typing import List, Dict, Any
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
import logging
import time
import numpy as np
from openai import OpenAI as OpenAIClient
from google.colab import userdata
from langchain.schema import Document

# 로깅 설정
logging.basicConfig(level=logging.INFO)

# OpenAI API 키 설정
openai_api_key = userdata.get('FINAL_TEAM3')
if not openai_api_key:
    raise ValueError("OpenAI API 키가 설정되지 않았습니다.")

# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_b"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# Langchain Qdrant 초기화
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embeddings=embeddings
)

# Sentence Transformer 모델
st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# OpenAI 클라이언트 초기화
openai_client = OpenAIClient(api_key=openai_api_key)

# 프롬프트 템플릿 정의
prompt_template = """
질문: {question}
관련 카테고리: {category}
관련 질병: {disease}
사용자 의도: {intent}
컨텍스트: {context}

위 정보를 바탕으로 답변해주세요:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["question", "category", "disease", "intent", "context"]
)

# OpenAI 클래스 초기화
llm = OpenAI(temperature=0.7, openai_api_key=openai_api_key)

# RetrievalQA 체인 생성
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

def get_unique_metadata():
    try:
        response = client.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=None,
            limit=10000,
            with_payload=True,
            with_vectors=False
        )

        categories = set()
        diseases = set()
        intents = set()
        departments = set()

        for point in response[0]:
            category = point.payload.get("질명_카테고리")
            disease = point.payload.get("질병")
            intent = point.payload.get("의도")
            department = point.payload.get("부서")

            if category:
                categories.add(category)
            if disease:
                diseases.add(disease)
            if intent:
                intents.add(intent)
            if department:
                departments.add(department)

        return list(categories), list(diseases), list(intents), list(departments)
    except Exception as e:
        logging.error(f"메타데이터 가져오기 중 오류 발생: {str(e)}")
        return [], [], [], []

DISEASE_CATEGORIES, DISEASES, INTENTS, DEPARTMENTS = get_unique_metadata()

def extract_metadata(query, categories, diseases, intents, departments):
    query_lower = query.lower()
    category = next((cat for cat in categories if cat.lower() in query_lower), None)
    disease = next((dis for dis in diseases if dis.lower() in query_lower), None)
    intent = next((intent for intent in intents if intent.lower() in query_lower), None)
    department = next((dept for dept in departments if dept.lower() in query_lower), None)
    logging.debug(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}, 부서={department}")
    return category, disease, intent, department

def expand_query(query: str) -> List[str]:
    expanded_queries = [query]

    synonyms = {
        "증상": ["징후", "병증", "증세"],
        "치료": ["처치", "요법", "치료법"],
        "예방": ["방지", "예방책", "예방법"],
        "통증": ["아픔", "고통", "불편함"],
    }

    words = query.split()
    for word in words:
        if word in synonyms:
            for syn in synonyms[word]:
                expanded_queries.append(query.replace(word, syn))

    category, disease, intent, department = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS, DEPARTMENTS)
    if category:
        expanded_queries.append(f"{query} {category}")
    if disease:
        expanded_queries.append(f"{query} {disease}")
    if intent:
        expanded_queries.append(f"{query} {intent}")
    if department:
        expanded_queries.append(f"{query} {department}")

    logging.debug(f"확장된 쿼리: {expanded_queries}")
    return list(set(expanded_queries))

def bm25_search(query: str, documents: List[Document], top_k: int = 5) -> List[Document]:
    if not documents:
        logging.warning("BM25 검색: 문서 목록이 비어 있습니다.")
        return []
    try:
        corpus = [doc.page_content for doc in documents if doc.page_content.strip()]
        if not corpus:
            logging.warning("BM25 검색: 모든 문서의 내용이 비어 있습니다.")
            return []
        bm25 = BM25Okapi(corpus)
        scores = bm25.get_scores(query.split())
        top_results = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)[:top_k]
        return [doc for doc, score in top_results]
    except Exception as e:
        logging.error(f"BM25 검색 중 오류 발생: {str(e)}")
        return []

def vector_search(query, documents, top_k=5):
    if not documents:
        logging.warning("벡터 검색: 문서 목록이 비어 있습니다.")
        return []
    query_embedding = st_model.encode([query])[0]
    doc_embeddings = st_model.encode([doc.page_content for doc in documents])
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_results = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)[:top_k]
    return [doc for doc, score in top_results]

def ensemble_search(query: str, documents: List[Document], top_k: int = 5) -> List[Document]:
    if not documents:
        logging.warning("앙상블 검색: 문서 목록이 비어 있습니다.")
        return []

    bm25_results = bm25_search(query, documents, top_k)
    vector_results = vector_search(query, documents, top_k)

    combined_results = list(dict.fromkeys(bm25_results + vector_results))

    if not combined_results:
        logging.warning("앙상블 검색: 유효한 결과가 없습니다.")
        return []

    bm25 = BM25Okapi([doc.page_content for doc in combined_results])
    scores = bm25.get_scores(query.split())
    sorted_results = sorted(zip(combined_results, scores), key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in sorted_results[:top_k]]

def generate_gpt4_response(query: str, context: str = None, metadata: Dict[str, str] = None) -> str:
    try:
        system_message = "You are a helpful assistant specializing in medical information. Provide accurate and concise answers based on the given context and metadata."
        user_message = f"Question: {query}\n\nContext: {context or 'No context available'}\n\nMetadata: {metadata or 'No metadata available'}"

        response = openai_client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            max_tokens=150,
            n=1,
            stop=None,
            temperature=0.7,
        )

        return response.choices[0].message.content.strip()
    except Exception as e:
        logging.error(f"GPT-4 응답 생성 중 오류 발생: {str(e)}")
        return f"죄송합니다. GPT-4 응답을 생성하는 중에 오류가 발생했습니다: {str(e)}"

def process_query(query: str) -> str:
    start_time = time.time()

    logging.info(f"처리 중인 쿼리: {query}")
    category, disease, intent, department = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS, DEPARTMENTS)
    logging.info(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}, 부서={department}")

    expanded_queries = expand_query(query)
    logging.info(f"확장된 쿼리: {expanded_queries}")

    try:
        all_documents = vector_store.similarity_search(query, k=20)

        # Qdrant 결과를 Document 객체로 변환
        valid_documents = [
            Document(page_content=doc.payload.get('답변', ''),
                     metadata={
                         '질병_카테고리': doc.payload.get('질명_카테고리'),
                         '질병': doc.payload.get('질병'),
                         '의도': doc.payload.get('의도'),
                         '부서': doc.payload.get('부서')
                     })
            for doc in all_documents if doc.payload.get('답변')
        ]

        if not valid_documents:
            logging.warning("Qdrant에서 유효한 검색 결과가 없습니다.")
            return generate_gpt4_response(query)

        ensemble_results = ensemble_search(query, valid_documents, top_k=5)

        if not ensemble_results:
            logging.warning("앙상블 검색 결과가 없습니다.")
            return generate_gpt4_response(query)

        best_match = ensemble_results[0]
        context = best_match.page_content
        metadata = best_match.metadata

        end_time = time.time()
        processing_time = end_time - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        return generate_gpt4_response(query, context=context, metadata=metadata)
    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}")
        return generate_gpt4_response(query)

def evaluate_system():
    test_queries = [
        "감기에 걸렸을 때 어떻게 해야 하나요?",
        "고혈압의 증상은 무엇인가요?",
        "당뇨병 예방을 위한 식습관은?"
    ]

    total_time = 0
    for query in test_queries:
        start_time = time.time()
        response = process_query(query)
        end_time = time.time()

        processing_time = end_time - start_time
        total_time += processing_time

        print(f"질문: {query}")
        print(f"답변: {response}")
        print(f"처리 시간: {processing_time:.2f}초")
        print("-" * 50)

    avg_time = total_time / len(test_queries)
    print(f"평균 처리 시간: {avg_time:.2f}초")

if __name__ == "__main__":
    evaluate_system()

TypeError: QdrantVectorStore.__init__() got an unexpected keyword argument 'embeddings'